# Notebook 08 — Ensemble Stacking

**Projekt:** Współzależność rynków aktywów cyfrowych i tradycyjnych systemów finansowych  

---

## Cel notebooka

Budujemy system **Ensemble Stacking** łączący 4 modele bazowe w meta-model.

### Architektura Stacking
```
Warstwa 1 (Modele bazowe):
  ┌────────────┐  ┌─────────┐  ┌───────────┐  ┌──────────┐
  │    LSTM    │  │   GRU   │  │  XGBoost  │  │ LightGBM │
  └──────┬─────┘  └────┬────┘  └─────┬─────┘  └────┬─────┘
         │              │             │              │
         └──────────────┴─────────────┴──────────────┘
                              │
                   + autoencoder_re
                              │
                        Meta-dataset
                              │
                  ┌───────────────────────┐
                  │  Meta-learner         │
                  │  (Logistic Regression)│
                  └───────────┬───────────┘
                              │
                        HRI ∈ [0,1]
```

### Kluczowe właściwości
- Meta-learner uczy się **optymalnych wag** dla każdego modelu bazowego
- Korelacja między modelami bazowymi powinna być niska (komplementarność)
- OOF predictions gwarantują brak data leakage w meta-datasecie

In [1]:
# ============================================================
# Konfiguracja wspolna
# ============================================================
import os, warnings
os.environ.setdefault('KERAS_BACKEND', 'torch')
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
warnings.filterwarnings('ignore')
import importlib
import pandas as pd, numpy as np
import pipeline_lib as pl, models_lib as ml, run_analysis as ra
for m in (pl, ml, ra): importlib.reload(m)
pd.set_option('display.width', 160)
print(f'Ziarno losowe: {pl.SEED} | horyzont: {pl.HORIZON} sesji | embargo: {pl.EMBARGO} sesji')


Ziarno losowe: 42 | horyzont: 5 sesji | embargo: 10 sesji


In [2]:
# ============================================================
# Warstwa meta: warianty agregacji prognoz bazowych
# ============================================================
oof, y, master = ra.load()
cmp = ra.model_comparison(oof, y)
print(cmp.to_string())
print('\nWagi stackingu z ograniczeniem nieujemnosci w kolejnych foldach:')
print(pd.read_csv('results/tables/08_meta_weights.csv').to_string(index=False))


                                        accuracy  precision  recall      f1  auc_roc  auc_pr  auc_pr_lift   brier  threshold    n  n_positive  tp   fp  fn   tn
model                                                                                                                                                          
LSTM                                      0.8029     0.0932  0.8824  0.1685   0.8365  0.3421       15.113  0.0414     0.0508  751          17  15  146   2  588
GRU                                       0.6445     0.0504  0.8235  0.0949   0.7535  0.3055       13.496  0.0356     0.0117  751          17  14  264   3  470
XGBoost                                   0.9601     0.2759  0.4706  0.3478   0.8326  0.4124       18.218  0.0199     0.2366  751          17   8   21   9  713
LightGBM                                  0.9561     0.2500  0.4706  0.3265   0.8103  0.4385       19.371  0.0219     0.2538  751          17   8   24   9  710
Zespół (stacking z wagami nieujemnymi)  